# Tutorial 7.1: Optimisation foundations for building operation

This tutorial formulates a one-day heat-pump schedule using the verified $1R1C$ model. We compare hard and soft comfort constraints, investigate flat and dynamic electricity prices, verify the schedule in a perfectly matched model, and then introduce indoor-temperature measurement errors.

## Learning outcomes

By the end of the tutorial, you should be able to:

1. distinguish optimisation variables from known parameters;
2. implement hard and soft operating constraints in CVXPY;
3. diagnose feasibility and verify a returned solution; and
4. explain how prices and initial-temperature measurement errors affect operation.

## Indicative schedule

- formulation and hard/soft constraints: 25 minutes;
- flat and dynamic tariff comparison: 15 minutes;
- matched-model checks: 15 minutes;
- measurement-error experiment: 20 minutes;
- interpretation and discussion: 15 minutes.

## 1. Set-up and one-day case

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    raise FileNotFoundError("Run this notebook from the repository root.")

plt.rcParams.update({"figure.figsize": (10, 4), "figure.dpi": 110})

In [ ]:
conditions = pd.read_csv(
    ROOT / "data" / "reference_building" / "operation_conditions.csv",
    index_col="timestamp",
    parse_dates=True,
).iloc[:48]
model = json.loads(
    (ROOT / "data" / "reference_building" / "verified_controller_model.json").read_text()
)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
axes[0].plot(conditions["outdoor_temperature_c"], label="Outdoor temperature")
axes[0].plot(conditions["comfort_min_c"], label="Lower comfort bound")
axes[0].plot(conditions["comfort_max_c"], label="Upper comfort bound")
axes[0].set_ylabel("Temperature [°C]")
axes[0].legend()
axes[1].step(conditions.index, conditions["import_price_gbp_per_kwh"], where="post")
axes[1].set_ylabel("Import price [£/kWh]")
axes[1].set_xlabel("Time")
plt.show()

## 2. Task 1 — formulate the scheduling problem

The decision variables are electrical heat-pump power $P_{el,k}$, predicted temperature $T_k$ and lower comfort slack $s_k$. The model uses $q_{th,k}=\mathrm{COP}\,P_{el,k}$.

A hard comfort constraint must be satisfied exactly:

$$
T_k \geq T_k^{min}.
$$

A soft constraint introduces nonnegative slack $s_k$:

$$
T_k+s_k \geq T_k^{min}, \qquad s_k \geq 0.
$$

The soft-constraint objective is

$$
J=\sum_k p_kP_{el,k}\Delta t+\rho\sum_k s_k\Delta t.
$$

Hard constraints can make a problem infeasible. Soft constraints preserve feasibility when sufficient slack is allowed, but a low penalty $\rho$ can make avoidable discomfort economically attractive. Slack must therefore be penalised and reported rather than hidden.

In [ ]:
def optimise_heating(
    conditions,
    model,
    initial_temperature_c=20.0,
    soft_constraints=True,
    slack_penalty=25.0,
):
    """Return an optimal electrical-power and predicted-temperature schedule."""
    n_steps = len(conditions)
    dt = model["sampling_interval_hours"]

    # TODO 1.1: create N nonnegative power and slack variables and N+1 temperatures.
    # TODO 1.2: add the initial condition, capacity and dynamics constraints.
    # TODO 1.3: use slack in the lower comfort bound only for soft constraints;
    #           for hard constraints, impose the bound without slack.
    # TODO 1.4: minimise energy cost plus the comfort-slack penalty.
    # TODO 1.5: solve, check problem.status and return costs and arrays.
    raise NotImplementedError("Formulate and solve the heating schedule.")

## 3. Task 2 — compare hard and soft constraints

The normal case is feasible with hard constraints. For the soft case, use $\rho=0.25$ £/(°C h) so that the energy–comfort trade-off is visible. This deliberately low teaching value is not a recommended operational setting.

In [ ]:
true_initial_temperature_c = 20.0
soft_slack_penalty = 0.25

hard_schedule = optimise_heating(
    conditions,
    model,
    initial_temperature_c=true_initial_temperature_c,
    soft_constraints=False,
)
soft_schedule = optimise_heating(
    conditions,
    model,
    initial_temperature_c=true_initial_temperature_c,
    soft_constraints=True,
    slack_penalty=soft_slack_penalty,
)

schedule_comparison = pd.DataFrame(
    {
        "Hard": {
            "Energy cost [£]": hard_schedule["energy_cost_gbp"],
            "Slack penalty [£]": hard_schedule["slack_cost_gbp"],
            "Predicted slack [°C h]": hard_schedule["comfort_slack_c"].sum()
            * model["sampling_interval_hours"],
        },
        "Soft": {
            "Energy cost [£]": soft_schedule["energy_cost_gbp"],
            "Slack penalty [£]": soft_schedule["slack_cost_gbp"],
            "Predicted slack [°C h]": soft_schedule["comfort_slack_c"].sum()
            * model["sampling_interval_hours"],
        },
    }
).T
display(schedule_comparison.round(3))

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8))
axes[0].step(conditions.index, conditions["import_price_gbp_per_kwh"], where="post")
axes[0].set_ylabel("Price [£/kWh]")
axes[1].step(
    conditions.index,
    hard_schedule["electric_power_kw"],
    where="post",
    label="Hard",
)
axes[1].step(
    conditions.index,
    soft_schedule["electric_power_kw"],
    where="post",
    label="Soft",
    alpha=0.8,
)
axes[1].set_ylabel("Heat-pump power [kW]")
axes[1].legend()
axes[2].plot(
    conditions.index,
    hard_schedule["predicted_temperature_c"][1:],
    label="Hard",
)
axes[2].plot(
    conditions.index,
    soft_schedule["predicted_temperature_c"][1:],
    label="Soft",
)
axes[2].plot(conditions["comfort_min_c"], "--", label="Comfort bounds", color="black")
axes[2].plot(conditions["comfort_max_c"], "--", color="black")
axes[2].set_ylabel("Temperature [°C]")
axes[2].set_xlabel("Time")
axes[2].legend()
plt.show()

## 4. Task 3 — compare flat and dynamic prices

Create a flat tariff using the mean dynamic price. Everything else—the weather, comfort bounds, initial condition and hard constraints—remains unchanged. Under a flat price, the optimiser has no financial reason to move heating between intervals. Under the dynamic tariff, thermal storage allows it to preheat before expensive periods. Evaluate both schedules under both tariffs so that the scheduling effect is separated from the tariff definition.

In [ ]:
flat_price_conditions = conditions.copy()
flat_price_conditions["import_price_gbp_per_kwh"] = conditions[
    "import_price_gbp_per_kwh"
].mean()

flat_price_schedule = optimise_heating(
    flat_price_conditions,
    model,
    initial_temperature_c=true_initial_temperature_c,
    soft_constraints=False,
)
dynamic_price_schedule = hard_schedule

tariff_comparison = pd.DataFrame(
    {
        "Flat tariff": {
            "Energy [kWh]": flat_price_schedule["electric_power_kw"].sum()
            * model["sampling_interval_hours"],
            "Cost under flat tariff [£]": flat_price_schedule["energy_cost_gbp"],
            "Cost under dynamic tariff [£]": float(
                np.sum(
                    flat_price_schedule["electric_power_kw"]
                    * conditions["import_price_gbp_per_kwh"].to_numpy()
                )
                * model["sampling_interval_hours"]
            ),
        },
        "Dynamic tariff": {
            "Energy [kWh]": dynamic_price_schedule["electric_power_kw"].sum()
            * model["sampling_interval_hours"],
            "Cost under flat tariff [£]": float(
                dynamic_price_schedule["electric_power_kw"].sum()
                * flat_price_conditions["import_price_gbp_per_kwh"].iloc[0]
                * model["sampling_interval_hours"]
            ),
            "Cost under dynamic tariff [£]": dynamic_price_schedule[
                "energy_cost_gbp"
            ],
        },
    }
).T
display(tariff_comparison.round(3))

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8))
axes[0].step(
    conditions.index,
    conditions["import_price_gbp_per_kwh"],
    where="post",
    label="Dynamic tariff",
)
axes[0].step(
    flat_price_conditions.index,
    flat_price_conditions["import_price_gbp_per_kwh"],
    where="post",
    label="Flat tariff",
)
axes[0].set_ylabel("Price [£/kWh]")
axes[0].legend()
axes[1].step(
    conditions.index,
    dynamic_price_schedule["electric_power_kw"],
    where="post",
    label="Dynamic tariff",
)
axes[1].step(
    conditions.index,
    flat_price_schedule["electric_power_kw"],
    where="post",
    label="Flat tariff",
    alpha=0.8,
)
axes[1].set_ylabel("Heat-pump power [kW]")
axes[1].legend()
axes[2].plot(
    conditions.index,
    dynamic_price_schedule["predicted_temperature_c"][1:],
    label="Dynamic tariff",
)
axes[2].plot(
    conditions.index,
    flat_price_schedule["predicted_temperature_c"][1:],
    label="Flat tariff",
)
axes[2].plot(conditions["comfort_min_c"], "--", color="black", label="Comfort bounds")
axes[2].plot(conditions["comfort_max_c"], "--", color="black")
axes[2].set_ylabel("Temperature [°C]")
axes[2].set_xlabel("Time")
axes[2].legend()
plt.show()

## 5. Task 4 — evaluate a perfectly matched model

We first remove every source of error. The controller model and evaluation model use exactly the same equation and parameters, the initial temperature is known exactly, and there is no process or measurement noise. Predicted and realised temperatures should therefore agree to numerical precision. This is a useful implementation check before studying uncertainty.

In [ ]:
def simulate_matched_model(
    conditions,
    model,
    electric_power_kw,
    initial_temperature_c=20.0,
):
    """Apply a fixed schedule using exactly the controller-model dynamics."""
    temperatures = [float(initial_temperature_c)]
    for step, electric_power in enumerate(electric_power_kw):
        current_temperature = temperatures[-1]
        delivered_heat = model["nominal_cop"] * electric_power
        next_temperature = (
            current_temperature
            + model["loss_coefficient_per_step"]
            * (conditions["outdoor_temperature_c"].iloc[step] - current_temperature)
            + model["heat_coefficient_c_per_kw_step"] * delivered_heat
        )
        temperatures.append(next_temperature)
    return pd.Series(temperatures[1:], index=conditions.index, name="temperature_c")


def calculate_schedule_metrics(conditions, model, electric_power_kw, temperature_c):
    """Return realised energy, cost and lower-comfort metrics."""
    dt = model["sampling_interval_hours"]
    electric_power_kw = np.asarray(electric_power_kw, dtype=float)
    temperature_c = np.asarray(temperature_c, dtype=float)
    lower_bound = conditions["comfort_min_c"].to_numpy()
    violation = np.maximum(lower_bound - temperature_c, 0.0)
    return {
        "Energy [kWh]": float(electric_power_kw.sum() * dt),
        "Cost [£]": float(
            np.sum(
                electric_power_kw
                * conditions["import_price_gbp_per_kwh"].to_numpy()
            )
            * dt
        ),
        "Violation [h]": float(np.sum(violation > 1e-6) * dt),
        "Degree-hours [°C h]": float(violation.sum() * dt),
        "Maximum deviation [°C]": float(violation.max()),
    }

In [ ]:
matched_results = {}
matched_metrics = {}
for constraint_name, schedule in {
    "Hard": hard_schedule,
    "Soft": soft_schedule,
}.items():
    realised_temperature = simulate_matched_model(
        conditions,
        model,
        schedule["electric_power_kw"],
        initial_temperature_c=true_initial_temperature_c,
    )
    matched_results[constraint_name] = realised_temperature
    predicted_temperature = schedule["predicted_temperature_c"][1:]
    maximum_mismatch = np.max(
        np.abs(realised_temperature.to_numpy() - predicted_temperature)
    )
    matched_metrics[constraint_name] = {
        **calculate_schedule_metrics(
            conditions,
            model,
            schedule["electric_power_kw"],
            realised_temperature,
        ),
        "Prediction mismatch [°C]": maximum_mismatch,
    }

matched_metric_table = pd.DataFrame(matched_metrics).T
display(matched_metric_table.round(4))
assert matched_metric_table["Prediction mismatch [°C]"].max() < 1e-8

## 6. Task 5 — sensitivity to measurement error

The true initial temperature remains 20 °C, but the optimiser receives

$$
T_0^{measured}=T_0^{true}+e_T.
$$

Use signed errors from −1 °C to +1 °C. A positive value means the sensor reads too warm, so the optimiser may schedule too little heat. Only the initial measurement enters this fixed open-loop schedule. Errors in later measurements would matter only if the controller recalculated its decisions, as in model predictive control.

In [ ]:
def run_measurement_error_experiment(
    conditions,
    model,
    measurement_errors_c,
    soft_slack_penalty=0.25,
    true_initial_temperature_c=20.0,
):
    """Compare hard and soft schedules for signed initial measurement errors."""
    # TODO 5.1: loop over hard and soft constraints and each measurement error.
    # TODO 5.2: optimise using measured temperature = true temperature + error.
    # TODO 5.3: simulate from the true temperature and calculate realised metrics.
    # TODO 5.4: return one tidy DataFrame containing every experiment.
    raise NotImplementedError("Run the measurement-error experiment.")

In [ ]:
measurement_errors_c = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
measurement_error_results = run_measurement_error_experiment(
    conditions,
    model,
    measurement_errors_c,
    soft_slack_penalty=soft_slack_penalty,
    true_initial_temperature_c=true_initial_temperature_c,
)
display(measurement_error_results.round(3))

plot_data = measurement_error_results.reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for constraint_name, group in plot_data.groupby("Constraint"):
    group = group.sort_values("Measurement error [°C]")
    axes[0].plot(
        group["Measurement error [°C]"],
        group["Cost [£]"],
        marker="o",
        label=constraint_name,
    )
    axes[1].plot(
        group["Measurement error [°C]"],
        group["Degree-hours [°C h]"],
        marker="o",
        label=constraint_name,
    )

axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Initial measurement error [°C]")
axes[0].set_ylabel("Realised cost [£]")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Initial measurement error [°C]")
axes[1].set_ylabel("Realised discomfort [°C h]")
axes[1].legend(title="Constraint")
plt.tight_layout()
plt.show()

### Interpretation

Read the measurement-error axis as `measured minus true`. Compare both the magnitude and direction of the error. A colder reading tends to produce a conservative schedule with more heating, whereas a warmer reading can reduce cost while increasing realised comfort violations.

## Discussion

1. When does the dynamic-tariff optimiser preheat, and why?
2. What is gained and lost when a hard comfort constraint is replaced by a soft one?
3. Why are predicted and realised temperatures identical in the zero-error experiment?
4. Why does a positive initial-temperature measurement error tend to create more discomfort?
5. How does the slack penalty affect the balance between cost and comfort?
6. Why would repeated measurement errors matter more for a controller that re-optimises during the day?